# runkk-writer QLoRA 학습 (S15P31A207-194)

분석 요약 → 코칭 문장(JSON)을 쓰는 작은 모델을 학습하고 GGUF(Q4_K_M)로 바꿔 Drive에 저장한다.

1. 런타임: **런타임 > 런타임 유형 변경 > A100 GPU** (없으면 L4. 그때는 아래 `BATCH`를 4로)
2. Drive `MyDrive/runkk-writer/data/`에 `train.jsonl`·`eval.jsonl`을 올려 둔다 (`ai/data/writer/`에서 만든 파일)
3. 위에서부터 차례로 실행한다. 결과는 `MyDrive/runkk-writer/runs/<RUN_NAME>/`에 쌓인다

| 결과 | 내용 |
|---|---|
| `lora/` | LoRA 어댑터 (다시 학습·병합용) |
| `gguf/*.gguf` | Q4_K_M 모델. EC2 llama.cpp 컨테이너가 읽는다(-196) |
| `eval_outputs.jsonl`, `eval_report.json` | 홀드아웃 100건 생성 결과와 자동 검사 통과율 |
| `loss.png`, `config.json` | 학습 손실 곡선과 설정 (노션 기록용) |

노트북을 git에 올릴 때는 출력을 지운다(**편집 > 모든 출력 지우기**).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
%%capture
!pip install -q unsloth

## 설정

In [ ]:
RUN_NAME = "v1-qwen3-4b"  # 바꿔 가며 여러 번 돌린다
MODEL = "unsloth/Qwen3-4B-Instruct-2507"  # 대안: "kakaocorp/kanana-nano-2.1b-instruct" (한국어, 더 작음)
MAX_SEQ = 2048
LORA_R = 16
EPOCHS = 2
LR = 2e-4
BATCH = 8  # A100 기준. L4·T4면 4
GRAD_ACCUM = 2
SEED = 7
GGUF_QUANT = "q4_k_m"

from google.colab import drive

drive.mount("/content/drive")

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/runkk-writer")
DATA = ROOT / "data"
OUT = ROOT / "runs" / RUN_NAME
OUT.mkdir(parents=True, exist_ok=True)
assert (DATA / "train.jsonl").exists() and (DATA / "eval.jsonl").exists(), (
    "Drive에 train.jsonl·eval.jsonl을 먼저 올린다"
)

## 모델 불러오기 (4bit) + LoRA

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL, max_seq_length=MAX_SEQ, load_in_4bit=True, dtype=None
)
model = FastLanguageModel.get_peft_model(
    model,
    r=LORA_R,
    lora_alpha=LORA_R,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=SEED,
)

## 데이터

In [ ]:
import json

from datasets import Dataset


def read_jsonl(path):
    return [json.loads(line) for line in open(path, encoding="utf-8") if line.strip()]


train_rows = read_jsonl(DATA / "train.jsonl")  # {"messages": [system, user, assistant]}
eval_rows = read_jsonl(DATA / "eval.jsonl")  # {"brief", "output", "rule", ...}


def to_text(row):
    return {"text": tokenizer.apply_chat_template(row["messages"], tokenize=False)}


train_ds = Dataset.from_list(train_rows).map(to_text, remove_columns=["messages"])
lengths = [len(tokenizer(t)["input_ids"]) for t in train_ds["text"]]
print(len(train_ds), "건, 토큰 길이 최대", max(lengths), "평균", sum(lengths) // len(lengths))
assert max(lengths) < MAX_SEQ
print(train_ds[0]["text"][-600:])

## 학습 (답변 부분만 손실 계산)

In [ ]:
from trl import SFTConfig, SFTTrainer
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=train_ds,
    args=SFTConfig(
        dataset_text_field="text",
        max_seq_length=MAX_SEQ,
        per_device_train_batch_size=BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        num_train_epochs=EPOCHS,
        learning_rate=LR,
        lr_scheduler_type="cosine",
        warmup_ratio=0.05,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        seed=SEED,
        output_dir="/content/outputs",
        report_to="none",
    ),
)
trainer = train_on_responses_only(
    trainer,
    instruction_part="<|im_start|>user\n",
    response_part="<|im_start|>assistant\n",
)
stats = trainer.train()
print(stats)

In [ ]:
import matplotlib.pyplot as plt

log = [x for x in trainer.state.log_history if "loss" in x]
plt.plot([x["step"] for x in log], [x["loss"] for x in log])
plt.xlabel("step")
plt.ylabel("loss")
plt.title(RUN_NAME)
plt.savefig(OUT / "loss.png", dpi=120)
plt.show()

config = {
    "run": RUN_NAME,
    "model": MODEL,
    "train": len(train_rows),
    "epochs": EPOCHS,
    "lr": LR,
    "lora_r": LORA_R,
    "batch": BATCH * GRAD_ACCUM,
    "final_loss": log[-1]["loss"] if log else None,
    "runtime_s": round(stats.metrics.get("train_runtime", 0)),
}
(OUT / "config.json").write_text(json.dumps(config, ensure_ascii=False, indent=2))
model.save_pretrained(str(OUT / "lora"))
tokenizer.save_pretrained(str(OUT / "lora"))
config

## 홀드아웃 100건 평가

ai 서버의 `runkk_ai/writer/checks.py`와 같은 검사로 통과율을 본다(아래 셀은 그 파일을 그대로 붙인 것).

In [ ]:
"""writer 출력 자동 검사. 학습 데이터 거르기와 평가(규칙 준수율, S15P31A207-195)가 같이 쓴다.

통과 기준
- JSON 하나, 키 4개(headline·good·fix·next), 빈 값 없음, 길이 상한
- 숫자: 출력의 모든 숫자가 brief(템플릿 문장 포함)에 있는 숫자여야 한다(숫자 지어내기 방지)
- 금칙어·이모지 없음
- all_good이 아닌데 고칠 점이 '없다'고 하지 않음
"""

import json
import re
from dataclasses import dataclass, field
from typing import Any

KEYS = ("headline", "good", "fix", "next")
MAX_LEN = {"headline": 25, "good": 120, "fix": 120, "next": 120}
FORBIDDEN = ("위험", "큰일", "병원", "진단", "부상을 입", "부상입", "질환", "반드시", "절대", "최악")
# 단위·순서처럼 요약에 없어도 자연스러운 작은 수(1주, 하루, 2번째 등)
FREE_NUMBERS = frozenset({0.0, 1.0, 2.0})
_NUMBER = re.compile(r"\d+(?:\.\d+)?")
_EMOJI = re.compile("[\U0001f300-\U0001faff☀-➿]")
_NO_FIX = re.compile(r"고칠 점(?:이|은)? (?:없|딱히 없)")


@dataclass
class CheckResult:
    ok: bool
    reasons: list[str] = field(default_factory=list)
    output: dict[str, str] | None = None


def numbers_in(text: str) -> set[float]:
    return {float(n) for n in _NUMBER.findall(text)}


def allowed_numbers(brief: dict[str, Any]) -> set[float]:
    return numbers_in(json.dumps(brief, ensure_ascii=False)) | FREE_NUMBERS


def parse_output(raw: str) -> dict[str, Any] | None:
    """앞뒤 군더더기(```json 등)를 걷어내고 JSON 객체 하나를 읽는다."""
    start, end = raw.find("{"), raw.rfind("}")
    if start < 0 or end <= start:
        return None
    try:
        value = json.loads(raw[start : end + 1])
    except json.JSONDecodeError:
        return None
    return value if isinstance(value, dict) else None


def check(raw: str, brief: dict[str, Any]) -> CheckResult:
    data = parse_output(raw)
    if data is None:
        return CheckResult(False, ["not_json"])
    reasons: list[str] = []
    if set(data) != set(KEYS):
        reasons.append("keys")
    out = {k: str(data.get(k) or "").strip() for k in KEYS}
    allowed = allowed_numbers(brief)
    for key in KEYS:
        text = out[key]
        if not text:
            reasons.append(f"empty:{key}")
            continue
        if len(text) > MAX_LEN[key]:
            reasons.append(f"too_long:{key}")
        if extra := numbers_in(text) - allowed:
            reasons.append(f"number:{key}:{','.join(sorted(f'{n:g}' for n in extra))}")
        if any(word in text for word in FORBIDDEN):
            reasons.append(f"forbidden:{key}")
        if _EMOJI.search(text):
            reasons.append(f"emoji:{key}")
    if brief.get("rule", {}).get("fix") != "all_good" and _NO_FIX.search(out["fix"]):
        reasons.append("fix_denied")
    return CheckResult(not reasons, reasons, out)

In [ ]:
import time
from collections import Counter

FastLanguageModel.for_inference(model)
SYSTEM = train_rows[0]["messages"][0]["content"]


def user_message(brief):
    return "분석 요약:\n" + json.dumps(brief, ensure_ascii=False, separators=(",", ":"))


results, reasons = [], Counter()
started = time.time()
for row in eval_rows:
    messages = [{"role": "system", "content": SYSTEM}, {"role": "user", "content": user_message(row["brief"])}]
    inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to("cuda")
    out = model.generate(input_ids=inputs, max_new_tokens=400, do_sample=False)
    raw = tokenizer.decode(out[0][inputs.shape[1] :], skip_special_tokens=True)
    r = check(raw, row["brief"])
    reasons.update(x.split(":")[0] for x in r.reasons)
    results.append(
        {
            "situationId": row["situationId"],
            "rule": row["rule"],
            "ok": r.ok,
            "reasons": r.reasons,
            "raw": raw,
            "teacher": row["output"],
            "template": row["brief"]["template"],
        }
    )

passed = sum(r["ok"] for r in results)
report = {
    "run": RUN_NAME,
    "eval": len(results),
    "passed": passed,
    "passRate": round(passed / len(results), 3),
    "reasons": dict(reasons),
    "secondsPerItem": round((time.time() - started) / len(results), 2),
    "byRule": dict(Counter(r["rule"]["fix"] for r in results if r["ok"])),
}
with open(OUT / "eval_outputs.jsonl", "w", encoding="utf-8") as f:
    for r in results:
        f.write(json.dumps(r, ensure_ascii=False) + "\n")
(OUT / "eval_report.json").write_text(json.dumps(report, ensure_ascii=False, indent=2))
report

In [ ]:
# 눈으로 몇 건 보기: 템플릿 / 선생(Opus) / 학생(이번 모델)
for r in results[:5]:
    print("==", r["rule"], "OK" if r["ok"] else r["reasons"])
    print("템플릿:", r["template"]["fix"])
    print("선생  :", r["teacher"]["fix"])
    print("학생  :", r["raw"][:400], "\n")

## GGUF(Q4_K_M)로 저장 → Drive

In [ ]:
gguf_dir = "/content/gguf"
model.save_pretrained_gguf(gguf_dir, tokenizer, quantization_method=GGUF_QUANT)

import glob
import shutil

(OUT / "gguf").mkdir(exist_ok=True)
for f in glob.glob(f"{gguf_dir}*/*.gguf") + glob.glob(f"{gguf_dir}/*.gguf"):
    if GGUF_QUANT.upper() in f.upper():
        shutil.copy(f, OUT / "gguf" / f"runkk-writer-{RUN_NAME}-{GGUF_QUANT}.gguf")
print(list((OUT / "gguf").iterdir()))